# Autonomous Agents: From Workflows to Self-Directed Systems

**What you'll learn:**
- The agent loop: LLM → Action → Environment Feedback → repeat
- Tool design with good ACI (Agent-Computer Interface)
- Implementing a working agent with transparency and guardrails
- When agents are appropriate vs. overkill
- Production-validated domains and patterns

**The leap:** In workflows, YOU write the control flow. In agents, the MODEL decides what to do next.

> *"Agents begin their work with either a command from, or interactive discussion with, the human user. Once the task is clear, agents plan and operate independently, potentially returning to the human for further clarification or judgement."*

**Prerequisites:** Complete notebooks 00-05 first, or be familiar with the five workflow patterns.

## The Agent Loop

![Autonomous agent loop — LLM Call cycles between Action and Feedback from Environment, with Human oversight and Stop conditions](assets/autonomous_agent_loop.webp)

**The core loop is remarkably simple:**

```
while not done and iterations < max_iterations:
    1. LLM decides next action (using tools + context)
    2. Execute the action in the environment
    3. Observe the result (environment feedback)
    4. LLM decides: continue, ask human, or stop?
```

**What makes this an agent (not a workflow):**
- The model chooses which tool to use at each step
- The number of steps is NOT predetermined
- The model can change strategy based on intermediate results
- It operates autonomously until hitting a stop condition

### The Three Principles Applied

| Principle | In Agent Context |
|-----------|-----------------|
| **Simplicity** | An agent is "just an LLM using tools in a loop" — don't over-engineer it |
| **Transparency** | Print each step's reasoning so humans can follow and intervene |
| **Careful ACI** | Tool design is where you spend MOST of your engineering time |

## When Agents Are Appropriate

Agents suit **open-ended problems** where:
- The required number of steps is unpredictable
- You cannot hardcode a fixed path  
- The LLM must operate for many turns
- The environment provides ground truth feedback at each step
- You have sufficient trust in the model's decision-making

### Validated Production Domains

| Domain | Why agents work here |
|--------|---------------------|
| **Customer support** | Natural conversation flow + tool access + measurable success (resolution) |
| **Coding agents** | Verifiable outputs (tests pass/fail) + structured problem space + iterative feedback |

Both share a critical property: **the environment gives clear feedback** (test results, customer satisfaction, tool outputs) that grounds the agent's decisions.

### When NOT to use agents

- Task has a fixed, known procedure → use a workflow pattern
- Single LLM call with good context handles it → don't build an agent
- No way to verify intermediate steps → errors compound silently
- Latency/cost constraints are tight → each loop iteration costs time and money

In [ ]:
import sys
import json
sys.path.append(".")
from util import llm_call, extract_xml

## Designing Tools with Good ACI

Before building the agent loop, we need tools. Tool design is where the ACI principles matter most.

**Anthropic's finding:** In building SWE-bench agents, more engineering time was spent on tool design than on the overall prompt.

### Good vs. Bad Tool Design

| Principle | Bad Design | Good Design |
|-----------|-----------|-------------|
| **Clear names** | `do_thing(x)` | `search_files(pattern, directory)` |
| **Obvious parameters** | `query(q, m, t)` | `query(search_term, max_results, timeout_seconds)` |
| **Poka-yoke** | Accepts relative paths | Requires absolute paths (no ambiguity) |
| **Documented** | No description | Includes examples, edge cases, boundaries |
| **LLM-friendly format** | Returns raw JSON blob | Returns formatted, readable text |

In [ ]:
# Define tools with GOOD ACI — clear names, descriptions, examples
TOOLS = {
    "search_knowledge": {
        "description": "Search a knowledge base for relevant information. Returns top matching passages.",
        "parameters": ["query"],
        "examples": ["search_knowledge('python async patterns')", "search_knowledge('error handling best practices')"],
        "function": lambda query: f"Results for '{query}':\n1. Use asyncio.gather() for concurrent tasks\n2. Always handle exceptions in async code with try/except\n3. Use asyncio.wait_for() to add timeouts to coroutines"
    },
    "calculate": {
        "description": "Perform mathematical calculations. Accepts any valid Python math expression.",
        "parameters": ["expression"],
        "examples": ["calculate('25 * 4 + 10')", "calculate('(100 / 3) ** 2')"],
        "function": lambda expr: str(eval(expr))
    },
    "write_file": {
        "description": "Write content to a file. Use absolute paths only. Creates parent directories if needed.",
        "parameters": ["path", "content"],
        "examples": ["write_file('/tmp/output.txt', 'Hello world')"],
        "function": lambda path, content: f"Written {len(content)} chars to {path}"
    },
    "read_file": {
        "description": "Read the contents of a file. Use absolute paths only.",
        "parameters": ["path"],
        "examples": ["read_file('/tmp/output.txt')"],
        "function": lambda path: f"File contents of {path}:\n# Example content\nThis is simulated file content for demonstration."
    },
    "done": {
        "description": "Signal that the task is complete. Provide the final answer/summary.",
        "parameters": ["summary"],
        "examples": ["done('Task completed: created 3 files with unit tests')"],
        "function": lambda summary: summary
    }
}

# Format tools for the agent prompt (applying ACI: document like onboarding)
def format_tools_for_prompt() -> str:
    """Create a clear tool reference for the agent."""
    lines = ["Available tools:\n"]
    for name, tool in TOOLS.items():
        lines.append(f"  {name}({', '.join(tool['parameters'])})")
        lines.append(f"    {tool['description']}")
        lines.append(f"    Example: {tool['examples'][0]}")
        lines.append("")
    return "\n".join(lines)

print(format_tools_for_prompt())

In [ ]:
def run_agent(task: str, max_iterations: int = 10, verbose: bool = True) -> str:
    """Run an autonomous agent loop with tool use.
    
    The agent:
    1. Receives a task
    2. Decides which tool to use (or signals done)
    3. Executes the tool and observes the result
    4. Repeats until done or max_iterations reached
    
    Args:
        task: The task to accomplish
        max_iterations: Safety limit to prevent infinite loops
        verbose: Whether to print each step (transparency principle)
    
    Returns:
        The agent's final summary
    """
    tools_description = format_tools_for_prompt()
    
    system_prompt = f"""You are an autonomous agent that accomplishes tasks by using tools.

{tools_description}

At each step, decide which tool to use and provide your reasoning.
When the task is complete, use the 'done' tool with a summary.

IMPORTANT: 
- Think step-by-step before acting
- Use the SIMPLEST approach that works
- If you're stuck, explain what's blocking you
- Signal 'done' as soon as the task is complete (don't over-iterate)

Respond in this format:
<reasoning>Your step-by-step thinking about what to do next</reasoning>
<tool>tool_name</tool>
<args>{{"param1": "value1"}}</args>"""

    history = []
    prompt = f"Task: {task}"
    
    if verbose:
        print(f"{'\u2550' * 60}")
        print(f"  AGENT TASK: {task}")
        print(f"{'\u2550' * 60}")
    
    for iteration in range(1, max_iterations + 1):
        if verbose:
            print(f"\n{'\u2500' * 60}")
            print(f"  Step {iteration}")
            print(f"{'\u2500' * 60}")
        
        # Agent decides next action
        full_prompt = prompt + "\n\n" + "\n".join(history) if history else prompt
        response = llm_call(full_prompt, system_prompt=system_prompt)
        
        reasoning = extract_xml(response, "reasoning")
        tool_name = extract_xml(response, "tool").strip()
        args_str = extract_xml(response, "args").strip()
        
        if verbose:
            print(f"  Reasoning: {reasoning[:200]}")
            print(f"  Action: {tool_name}({args_str})")
        
        # Execute the tool
        try:
            args = json.loads(args_str) if args_str else {}
        except json.JSONDecodeError:
            args = {"query": args_str}
        
        if tool_name not in TOOLS:
            result = f"Error: Unknown tool '{tool_name}'. Available: {list(TOOLS.keys())}"
        else:
            tool_fn = TOOLS[tool_name]["function"]
            try:
                result = tool_fn(**args)
            except Exception as e:
                result = f"Error executing {tool_name}: {e}"
        
        if verbose:
            print(f"  Result: {result[:200]}")
        
        # Check if done
        if tool_name == "done":
            if verbose:
                print(f"\n{'\u2550' * 60}")
                print(f"  \u2713 TASK COMPLETE (after {iteration} steps)")
                print(f"{'\u2550' * 60}")
                print(f"  Summary: {result}")
            return result
        
        # Add to history for next iteration
        history.append(f"Step {iteration}: Used {tool_name}({args_str}) \u2192 {result}")
    
    if verbose:
        print(f"\n\u26a0 Max iterations ({max_iterations}) reached without completion")
    return f"Incomplete after {max_iterations} iterations. Last result: {result}"

## Example 1: Research Agent

The agent investigates a question by searching for information, synthesizing findings, and deciding when it has enough context to answer. Notice how it directs its own workflow — the number of searches is determined by the model, not hardcoded.

In [ ]:
result = run_agent(
    "Research how connection pooling works in databases and explain it in 3 bullet points. "
    "Search for information, then provide a concise explanation.",
    max_iterations=5
)

## Guardrails for Autonomous Agents

Autonomous operation creates **compounding error risk** — each bad decision makes the next step worse. Essential guardrails:

| Guardrail | Implementation | Why |
|-----------|---------------|-----|
| **Max iterations** | `max_iterations` parameter | Prevents infinite loops and runaway costs |
| **Sandboxed environment** | Docker, temp directories, read-only access | Limits blast radius of mistakes |
| **Human-in-the-loop** | Pause for approval on high-stakes actions | Critical for irreversible operations |
| **Clear stop conditions** | `done` tool with summary | Agent knows when to stop |
| **Cost tracking** | Count API calls, estimate spend | Catch runaway agents early |
| **Transparency** | Print reasoning at each step | Humans can intervene when off-track |

### Cost Reality Check

```
Each iteration = 1 API call (~$0.003-0.015 for Sonnet)
A 10-step agent task = ~$0.03-0.15
A complex 50-step task = ~$0.15-0.75

Compare: A single workflow with 5 steps = ~$0.015-0.075
```

Agents are 2-10× more expensive than equivalent workflows. The flexibility must justify the cost.

## Production Agent Pattern: Coding Agent

![High-level coding agent flow — Human queries Interface, LLM clarifies task, searches files, writes code in test loop until passing](assets/coding_agent_flow.webp)

The coding agent is Anthropic's most validated agent architecture. Key properties:

1. **Clear feedback loop:** Write code → Run tests → See failures → Fix → Repeat
2. **Verifiable outputs:** Tests pass or fail (ground truth from environment)
3. **Structured problem space:** Code has rules — the agent can reason about correctness
4. **Natural tool set:** File read/write, test execution, search

### Why it works (connecting to our principles):

- **Simplicity:** "LLM + tools + loop" — no complex orchestration needed
- **Transparency:** Each code edit and test run is visible
- **ACI:** File operations with absolute paths, clear test output formats

In [ ]:
# Simulated coding agent with test verification
CODING_TOOLS = {
    "write_code": {
        "description": "Write Python code to a file.",
        "parameters": ["filename", "code"],
        "examples": ["write_code('solution.py', 'def add(a, b): return a + b')"],
        "function": lambda filename, code: f"Written to {filename}:\n{code}"
    },
    "run_tests": {
        "description": "Run test assertions against the code. Returns PASS or FAIL with details.",
        "parameters": ["test_code"],
        "examples": ["run_tests('assert add(2, 3) == 5')"],
        "function": lambda test_code: _run_test(test_code)
    },
    "done": {
        "description": "Signal task is complete with a summary.",
        "parameters": ["summary"],
        "examples": ["done('Implementation complete, all tests pass')"],
        "function": lambda summary: summary
    }
}

# Simulated test runner
_written_code = {}

def _run_test(test_code):
    """Simulate running tests — in production, this executes in a sandbox."""
    # Simulate: first attempt might have a bug
    if "fibonacci" in str(_written_code) and "memo" not in str(_written_code.get("code", "")):
        return "FAIL: test_fibonacci_large — RecursionError: maximum recursion depth exceeded"
    return "PASS: All 3 tests passed \u2713"

# Override TOOLS for this cell
original_tools = TOOLS.copy()
TOOLS.update(CODING_TOOLS)

result = run_agent(
    "Implement a fibonacci function that handles large inputs efficiently (fib(100)). "
    "Write the code, run tests, and fix any issues.",
    max_iterations=6
)

TOOLS = original_tools  # Restore

## Series Recap: The Complete Decision Framework

You've now learned the full spectrum from single calls to autonomous agents:

```
Single LLM Call
    │ (not enough? add complexity ↓)
    ▼
┌─────────────── WORKFLOWS ───────────────┐
│                                          │
│  Prompt Chaining    — Sequential steps   │
│  Routing            — Classify + dispatch│
│  Parallelization    — Fan-out + merge    │
│  Orchestrator-Workers — Dynamic decomp   │
│  Evaluator-Optimizer — Refine loop       │
│                                          │
└──────────────────────────────────────────┘
    │ (still not enough? need autonomy ↓)
    ▼
Autonomous Agent (LLM + tools + loop)
```

### The Decision Tree

1. **Does a single call work?** → Stop. You're done.
2. **Can you predefine the steps?** → Use a workflow.
3. **Are steps sequential?** → Prompt Chaining
4. **Are steps independent?** → Parallelization
5. **Need different handlers per input type?** → Routing
6. **Need dynamic decomposition?** → Orchestrator-Workers
7. **Need iterative improvement?** → Evaluator-Optimizer
8. **Steps unpredictable, need autonomy?** → Agent

### The Golden Rule

> **Start simple. Earn complexity. Measure everything.**

Every pattern in this series exists to solve a specific problem. If simpler approaches work, use them. Complexity is a cost, not a feature.

---

## Further Reading

- [Building Effective Agents](https://www.anthropic.com/research/building-effective-agents) — Anthropic's original research paper
- [Model Context Protocol](https://modelcontextprotocol.io/) — Standardized tool integration for agents
- [Claude Cookbooks](https://github.com/anthropics/claude-cookbooks) — More implementation examples

← Back to [README](README.md) | Start over at [00_foundations.ipynb](00_foundations.ipynb)